# Boltzmann partition function via logsumexp

log Z = logsumexp(−βE_i). After Tier B, the logsumexp block engages qsim via shift-by-max + Boltzmann sampling.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** thermodynamic partition function. **Classical:** sum of exponentials with shift-by-max. **Quantum (Tier B):** Boltzmann sampling — encode energies as rotation angles, measure.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx import ops
from uniqx.core import types
from uniqx.ops import expect

n = 4
rng = np.random.default_rng(103)
M = rng.standard_normal((n, n)) * 0.2
H = 0.5 * (M + M.T)
state = rng.standard_normal(2 * n) * 0.3

@trace
def prog(observable, st):
    e = expect(observable, st)
    return ops.logsumexp(e, result_type=types.scalar("f64"), axis=0)

module = prog(H.tolist(), state.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
